In [0]:
%run ./_bootstrap

In [0]:
import os
import json
import tempfile
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import mlflow
import mlflow.pyfunc
import joblib
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import pandas as pd

In [0]:
spark = SparkSession.builder.getOrCreate()
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA med")

In [0]:
# read latest classifier run id from registry table
latest = (
    spark.table("workspace.med.classifier_registry")
          .orderBy(F.col("created_at").desc())
          .limit(1).collect()
)

if not latest:
    raise RuntimeError("No rows in workspace.med.classifier_registry")

RUN_ID = latest[0]["classifier_run_id"]
BASE_MODEL_URI = f"runs:/{RUN_ID}/model"
print("Using RUN_ID:", RUN_ID)
print("BASE_MODEL_URI:", BASE_MODEL_URI)

In [0]:
# create temp dir for artifacts
tmpdir = tempfile.mkdtemp()
sk_path = os.path.join(tmpdir, "router_sklearn.joblib")
meta_path = os.path.join(tmpdir, "metadata.json")

# load trained sklearn pipeline from MLflow and save as a joblib artifact
sk_model = mlflow.sklearn.load_model(BASE_MODEL_URI)
joblib.dump(sk_model, sk_path)

serve_metadata = {
    "train_run_id": RUN_ID,
    "base_model_uri": BASE_MODEL_URI,
    "classes": list(getattr(sk_model.named_steps["logreg"], "classes_", [])),
}

# write meta data as json
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(serve_metadata, f)

print("tmpdir:", tmpdir)

In [0]:
# pyfunc wrapper returning label and confidence and routing decision
class RouterPyfunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        # load bundled artifacts
        self.model = joblib.load(context.artifacts["sk_model"])
        with open(context.artifacts["metadata"], "r", encoding="utf-8") as f:
            self.metadata = json.load(f) # load metadata json into memory

    def predict(self, context, model_input):
        # accept a dict payload or pandas df
        if isinstance(model_input, dict):
            df = pd.DataFrame([model_input])
        else:
            df = model_input.copy()

        if "question" not in df.columns:
            df["question"] = ""

        if "threshold" not in df.columns:
            df["threshold"] = 0.80

        questions = df["question"].astype(str).fillna("").tolist() # normalize questions into list of strings
        thresholds = df["threshold"].astype(float).fillna(0.80).tolist() # normalize thresholds into list of floats

        # Predict
        probs = self.model.predict_proba(questions) # get probabilities for each question
        classes = list(self.model.named_steps["logreg"].classes_) # get class names

        # build output for each question
        out_rows = []
        for i, q in enumerate(questions):
            if not q.strip():
                # return safe default if question is empty
                out_rows.append({
                    "label": None,
                    "confidence": 0.0,
                    "route": "all",
                    "probs": {},
                    "metadata": self.metadata,
                })
                continue

            p = probs[i] # proability for row i
            best_idx = int(p.argmax()) # get index of mot likely class
            label = classes[best_idx] # predict label
            conf = float(p[best_idx]) # get confidence for predicted label

            thr = float(thresholds[i])
            route = label if conf >= thr else "all"

            # build output
            out_rows.append({
                "label": label,
                "confidence": conf,
                "route": route,
                "probs": {classes[j]: float(p[j]) for j in range(len(classes))},
                "metadata": self.metadata,
            })

        return pd.DataFrame(out_rows)


In [0]:
input_example = pd.DataFrame([{"question": "what is ibuprofen used for", "threshold": 0.8}]) # example question for MLflow UI

# example output
sample_out = pd.DataFrame([{ 
    "label": "drug",
    "confidence": 0.95,
    "route": "drug",
    "probs": {"drug": 0.95, "condition": 0.05},
    "metadata": serve_metadata,
}])

signature = infer_signature(input_example, sample_out)

# log model
with mlflow.start_run(run_name="classifier_router_pyfunc_bundle") as run:
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=RouterPyfunc(),
        artifacts={"sk_model": sk_path, "metadata": meta_path},
        input_example=input_example,
        signature=signature,
        pip_requirements=["mlflow", "pandas", "numpy", "scikit-learn", "joblib"],
    )

    PYFUNC_URI = f"runs:/{run.info.run_id}/model"
    print("PYFUNC_URI:", PYFUNC_URI)